# 04b — RLS ABAC Option B: One policy + mapping table (separate table)

**Run after notebook 04** (governed tag from UI). This notebook is **Option B only**: everything in **one schema** **rls_abac_option_b** — division_access, fn_division_filter, sample_data, and the one policy. No mixing with 04 or rls_abac.

**Catalog:** `humana_payer`  
**Schema:** `rls_abac_option_b` only (division_access, fn_division_filter, sample_data — all in one schema)

---
**Documentation:** [ABAC](https://docs.databricks.com/data-governance/unity-catalog/abac/), [Microsoft tutorial](https://learn.microsoft.com/en-us/azure/databricks/data-governance/unity-catalog/abac/tutorial)

## Prerequisite

Create the **governed tag** `division` (UI Step 1 in notebook 04). No need for schema rls_abac — this notebook uses **only** schema **rls_abac_option_b**.

## Step 1: Create schema rls_abac_option_b (all Option B objects live here)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS humana_payer.rls_abac_option_b
COMMENT 'Option B only: division_access, fn_division_filter, sample_data';

## Step 2: Create division_access table + fn_division_filter UDF

In [0]:
%sql
CREATE OR REPLACE TABLE humana_payer.rls_abac_option_b.division_access (
  principal STRING COMMENT 'User email (current_user()) OR workspace group name (e.g. marketing, sales)',
  division  STRING COMMENT 'Allowed division value'
);
-- Use workspace group names so any user in that group gets access (recommended)
INSERT INTO humana_payer.rls_abac_option_b.division_access (principal, division) VALUES
  ('marketing', 'Marketing'),
  ('sales', 'Sales'),
  ('engineering', 'Engineering'),
  ('finance', 'Finance'),
  ('home', 'Home');
-- Or use user emails: (current_user(), 'Marketing'), ... for user-level access.

num_affected_rows,num_inserted_rows
5,5


**Principal = user or group:** The UDF treats `principal` as either a **user email** (matches `current_user()`) or a **workspace group name** (matches `is_member(principal)`). The inserts above use **group names** (marketing, sales, engineering, financee, home) so any user in those groups sees the matching rows. If you're not in any of those groups and need to test, run the cell below to add your user email for all divisions.

In [0]:
%sql
-- Add current user to division_access so you can see rows when testing (run once if you get 0 rows)
-- INSERT INTO humana_payer.rls_abac_option_b.division_access (principal, division) VALUES
--   (current_user(), 'Engineering'),
--   (current_user(), 'Marketing'),
--   (current_user(), 'Sales'),
--   (current_user(), 'Finance'),
--   (current_user(), 'Home');

In [0]:
# %sql
# -- principal can be a user email (match current_user()) OR a workspace group (is_member(principal))
# CREATE OR REPLACE FUNCTION humana_payer.rls_abac_option_b.fn_division_filter(division STRING)
# RETURNS BOOLEAN
# LANGUAGE SQL
# RETURN EXISTS (
#   SELECT 1 FROM humana_payer.rls_abac_option_b.division_access a
#   WHERE (a.principal = current_user() OR is_member(a.principal))
#     AND a.division = division
# );

In [0]:
%sql
CREATE OR REPLACE FUNCTION humana_payer.rls_abac_option_b.fn_division_filter(div_value STRING)
RETURNS BOOLEAN
LANGUAGE SQL
RETURN EXISTS (
  SELECT 1 FROM humana_payer.rls_abac_option_b.division_access a
    WHERE a.division = div_value
      AND (a.principal = current_user() OR is_member(a.principal))
);

## Step 3: Create sample_data table (policy applies to this table)

In [0]:
%sql
CREATE OR REPLACE TABLE humana_payer.rls_abac_option_b.sample_data (
  record_id STRING,
  division STRING,
  metric_value DOUBLE
);
INSERT INTO humana_payer.rls_abac_option_b.sample_data VALUES
  ('B001', 'Engineering', 110.0),
  ('B002', 'Marketing', 220.0),
  ('B003', 'Sales', 330.0),
  ('B004', 'Finance', 440.0),
  ('B005', 'Home', 550.0);

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
ALTER TABLE humana_payer.rls_abac_option_b.sample_data
  ALTER COLUMN division SET TAGS ('division' = 'Engineering');

## Step 4: Grant access

In [0]:
%sql
GRANT USAGE ON SCHEMA humana_payer.rls_abac_option_b TO `account users`;
GRANT SELECT ON TABLE humana_payer.rls_abac_option_b.division_access TO `account users`;
GRANT EXECUTE ON FUNCTION humana_payer.rls_abac_option_b.fn_division_filter TO `account users`;
GRANT SELECT ON TABLE humana_payer.rls_abac_option_b.sample_data TO `account users`;

## Step 5: UI — Create one policy (scope = rls_abac_option_b only)

**Catalog** → **humana_payer** → **Policies** → **New policy**.

- **Name:** `division_filter`  
- **Applied to:** e.g. marketing, sales, engineering, finance (or leave broad); UDF + division_access decide visibility.  
- **Except for:** workspace admins.  
- **Scope:** catalog humana_payer, schema **rls_abac_option_b**.  
- **Purpose:** Hide table rows.  
- **Conditions:** Select function **humana_payer.rls_abac_option_b.fn_division_filter**.  
- **Function parameters:** Map column by tag → division : **Engineering** (matches column tag above).

Keep **division_access** in sync with your groups; then you never add more UDFs or policies for new divisions.

## Step 6: Verify

**Workspace admins:** If you added yourself to the policy's **Except for** (e.g. "workspace admins"), you should see all rows without needing any `division_access` entries. If you still see no rows, check that (1) your user is in the exact group listed in **Except for** (some workspaces use "admins" or a custom group name), and (2) the policy is saved and the table scope is `humana_payer.rls_abac_option_b.sample_data`.

**Other users:** The policy only shows rows where `division_access` has `principal = current_user()` for that row's division. Run the optional cell above (after Step 2) to add your user to `division_access` for all divisions, then re-run the query below.

In [0]:
%sql
SELECT * FROM humana_payer.rls_abac_option_b.sample_data;

record_id,division,metric_value
B001,Engineering,110.0
B002,Marketing,220.0
B003,Sales,330.0
B004,Finance,440.0
B005,Home,550.0
